# Train SFW-SwinCBM on Google Colab
Notebook nay clone branch `ai_core`, train tren Defactify Hugging Face streaming, luu checkpoint vao Google Drive.


## 1. Check GPU
Runtime > Change runtime type > GPU. T4 la cau hinh mac dinh hop ly.


In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())


## 2. Mount Google Drive
Drive dung de luu checkpoint/output va cache nhe cho Hugging Face.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Clone repo branch ai_core
Cell nay dung Python thuan de tranh loi current directory bi xoa khi clone lai repo.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/chtr302/ai_generated_image_detection.git'
BRANCH = 'ai_core'
REPO_DIR = Path('/content/ai_generated_image_detection')

os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    'git', 'clone',
    '--single-branch',
    '--branch', BRANCH,
    REPO_URL,
    str(REPO_DIR),
], check=True)
os.chdir(REPO_DIR)
current_branch = subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip()
if current_branch != BRANCH:
    raise RuntimeError(f'Expected branch {BRANCH}, got {current_branch}')
print('cwd:', Path.cwd())
print('branch:', current_branch)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)


## 4. Install dependencies
Colab co san torch/torchvision. Cell nay cai them datasets va Pillow neu thieu.


In [ ]:
import os
from pathlib import Path

REPO_DIR = Path('/content/ai_generated_image_detection')
if not REPO_DIR.exists():
    raise FileNotFoundError(f'Repo chua clone thanh cong: {REPO_DIR}')
os.chdir(REPO_DIR)
print('cwd:', Path.cwd())

!python -m pip install -q datasets pillow tqdm


## 5. Verify repo version
Kiem tra code tren Colab da co cac tham so train moi hay chua. Neu fail, can push branch `ai_core` len GitHub roi restart runtime.


In [ ]:
import subprocess
import sys

help_result = subprocess.run(
    [sys.executable, '-m', 'src.model.train', '--help'],
    check=True,
    capture_output=True,
    text=True,
)
required_args = ['--hf-dataset', '--hf-shuffle-buffer', '--max-train-steps', '--max-val-steps', '--log-every']
missing = [arg for arg in required_args if arg not in help_result.stdout]
if missing:
    subprocess.run(['git', 'branch', '--show-current'], check=False)
    subprocess.run(['git', 'log', '-1', '--oneline'], check=False)
    raise RuntimeError('Repo tren Colab dang la code cu, thieu args: ' + ', '.join(missing))
print('OK: train.py supports Colab streaming args')
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)


## 6. Set data and output paths
Mac dinh dung Defactify tren Hugging Face. Neu muon dung data local tren Drive, gan `DATA_ROOT = Path('/content/drive/MyDrive/ai_data')`.


In [ ]:
from pathlib import Path

HF_DATASET = 'Rajarshi-Roy-research/Defactify_Image_Dataset'
HF_CACHE_DIR = Path('/content/hf_cache')  # Local Colab disk is faster than Google Drive for training reads.
HF_SHUFFLE_BUFFER = 10000
HF_NO_STREAMING = True  # Download/cache the dataset first, then train faster than streaming.

DATA_ROOT = None
OUTPUT_DIR = Path('/content/drive/MyDrive/ai_detector_outputs')

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def build_data_arg():
    if DATA_ROOT is not None:
        if not Path(DATA_ROOT).exists():
            raise FileNotFoundError(f'DATA_ROOT khong ton tai: {DATA_ROOT}')
        return f'--data-root {DATA_ROOT}'

    args = f'--hf-dataset {HF_DATASET} --hf-cache-dir {HF_CACHE_DIR} --hf-shuffle-buffer {HF_SHUFFLE_BUFFER}'
    if HF_NO_STREAMING:
        args += ' --hf-no-streaming'
    return args

print('HF_NO_STREAMING:', HF_NO_STREAMING)
print('HF_CACHE_DIR:', HF_CACHE_DIR)
print('data args:', build_data_arg())
print('OUTPUT_DIR:', OUTPUT_DIR)


## 7. Train config
Cau hinh nay uu tien toc do tren Colab: cache dataset vao local disk, image 224, batch 64, progress bar theo epoch.

In [ ]:
IMAGE_SIZE = 224  # 224 is faster. Use 256 only when you need a stronger final run.
BATCH_SIZE = 64  # If T4 does not OOM, try 128. If OOM, reduce to 32.
TRAIN_IMAGE_COUNT = 42000
TRAIN_STEPS_PER_EPOCH = (TRAIN_IMAGE_COUNT + BATCH_SIZE - 1) // BATCH_SIZE

TRAIN_CFG = {
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': 10,
    'grad_accum': 1,
    'nec': 10,
    'amp': 'fp16',
    'num_workers': 2,
    'max_train_steps': TRAIN_STEPS_PER_EPOCH,
    'max_val_steps': 2,
    'log_every': 20,
    'progress': 'bar',
}

print('IMAGE_SIZE:', IMAGE_SIZE)
print('BATCH_SIZE:', BATCH_SIZE)
print('TRAIN_STEPS_PER_EPOCH:', TRAIN_STEPS_PER_EPOCH)
print('Train config:', TRAIN_CFG)


def build_train_cli_args(cfg, resume=None):
    cli_parts = [
        build_data_arg(),
        f"--output-dir {OUTPUT_DIR}",
        f"--image-size {cfg['image_size']}",
        f"--batch-size {cfg['batch_size']}",
        f"--epochs {cfg['epochs']}",
        f"--grad-accum {cfg['grad_accum']}",
        f"--nec {cfg['nec']}",
        f"--amp {cfg['amp']}",
        f"--num-workers {cfg['num_workers']}",
        f"--max-val-steps {cfg['max_val_steps']}",
        f"--log-every {cfg['log_every']}",
        f"--progress {cfg['progress']}",
    ]
    if cfg.get('max_train_steps') is not None:
        cli_parts.append(f"--max-train-steps {cfg['max_train_steps']}")
    if resume is not None:
        cli_parts.append(f"--resume {resume}")
    return ' '.join(cli_parts)


## 8. Train model
Chi co mot cell train. Sau moi epoch chi in loss/accuracy/precision/recall/F1/balanced accuracy.

In [ ]:
train_args = build_train_cli_args(TRAIN_CFG)
!python -m src.model.train {train_args}


## 9. Benchmark test 100 images
Benchmark dung `test` split va gioi han 100 anh de so sanh cong bang. Neu muon so sanh voi LLM/model bai bao, hay chay model do tren cung 100 anh roi luu JSON dang: `[{'model': 'GPT-4o prompt', 'sample_count': 100, 'accuracy': 0.0, 'f1_ai': 0.0, 'note': 'same test images'}]`. Khong nen copy accuracy tu paper neu paper khong dung cung 100 anh nay.

In [ ]:
BENCHMARK_SAMPLES = 100
CHECKPOINT = OUTPUT_DIR / 'best.pt'
COMPARE_JSON = None  # Vi du: Path('/content/drive/MyDrive/external_model_100.json')

if not CHECKPOINT.exists():
    raise FileNotFoundError(f'Khong tim thay checkpoint best.pt: {CHECKPOINT}')

benchmark_args = ' '.join([
    build_data_arg(),
    f"--checkpoint {CHECKPOINT}",
    "--split test",
    f"--max-samples {BENCHMARK_SAMPLES}",
    f"--image-size {IMAGE_SIZE}",
    f"--batch-size {BATCH_SIZE}",
    f"--num-workers {TRAIN_CFG['num_workers']}",
    f"--nec {TRAIN_CFG['nec']}",
    f"--amp {TRAIN_CFG['amp']}",
])
if COMPARE_JSON is not None:
    benchmark_args += f" --compare-json {COMPARE_JSON}"

!python -m src.model.benchmark {benchmark_args}


## 10. Test thuc te AI
Output chinh khi test: `prediction`, `ai_probability`, `real_probability`, `confidence`, `concepts_top5`, `explanation`, `fft_explanation`. `fft_explanation` dien giai theo cach con nguoi: FFT khong nhin noi dung vat the, ma nhin dau vet tan so/texture/lap mau bat thuong de ho tro quyet dinh AI hay real.

In [ ]:
IMAGE_PATH = Path('/content/drive/MyDrive/sample.jpg')
CHECKPOINT = OUTPUT_DIR / 'best.pt'

if not IMAGE_PATH.exists():
    raise FileNotFoundError(f'Khong tim thay anh test: {IMAGE_PATH}')
if not CHECKPOINT.exists():
    raise FileNotFoundError(f'Khong tim thay checkpoint best.pt: {CHECKPOINT}')

!python -m src.model.inference --image {IMAGE_PATH} --checkpoint {CHECKPOINT} --image-size {IMAGE_SIZE} --nec {TRAIN_CFG['nec']}
